## Load Libraries

In [ ]:
import pandas as pd
import numpy as np

## Load Data into Data Frame

In [ ]:
df = pd.read_csv("DOHMH_New_York_City_Restaurant_Inspection_Results_20260307.csv")

In [ ]:
df.shape
df.info()
df.head()
df.nunique().sort_values()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296647 entries, 0 to 296646
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  296647 non-null  int64  
 1   DBA                    296645 non-null  object 
 2   BORO                   296647 non-null  object 
 3   BUILDING               295688 non-null  object 
 4   STREET                 296646 non-null  object 
 5   ZIPCODE                293468 non-null  float64
 6   PHONE                  296633 non-null  object 
 7   CUISINE DESCRIPTION    293340 non-null  object 
 8   INSPECTION DATE        296647 non-null  object 
 9   ACTION                 293340 non-null  object 
 10  VIOLATION CODE         291042 non-null  object 
 11  VIOLATION DESCRIPTION  291041 non-null  object 
 12  CRITICAL FLAG          296647 non-null  object 
 13  SCORE                  280013 non-null  float64
 14  GRADE                  146232 non-nu

,0
RECORD DATE,1
CRITICAL FLAG,3
ACTION,5
BORO,6
GRADE,6
INSPECTION TYPE,34
Council District,51
Community Board,69
CUISINE DESCRIPTION,90
SCORE,145


## 1. Drop irrelevant columns

In [ ]:
df = df.drop(["BIN",	"BBL",	"NTA",	"Location", "Community Board", "Council District", "Census Tract"], axis=1)

In [ ]:
df.head(5)

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude
0,50169247,WILDLIFE CONSERVATION SOCIETY,Bronx,2300,SOUTHERN BOULEVARD,10460.0,7187418170,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/06/2026,NaN,40.850537,-73.882455
1,50180337,BAO'S PASTRY LLC,Queens,135-25,ROOSEVELT AVENUE,11354.0,518526666,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/06/2026,NaN,40.759150,-73.831345
2,50173810,EARTHBAR,Manhattan,10,EAST 53 STREET,10022.0,6467070033,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/06/2026,NaN,40.760114,-73.975143
3,50182209,BUTTER CHICKEN SOCIAL,Manhattan,424,5 AVENUE,10018.0,5513259481,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/06/2026,NaN,40.751049,-73.982665
4,50122548,FAUZIA'S HEAVENLY DELIGHTS INC,Manhattan,517,CLAYTON AVE,NaN,7189305711,NaN,01/01/1900,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN,03/06/2026,NaN,0.000000,0.000000


In [ ]:
# Convert types early
date_cols = ["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [ ]:
# Handle placeholder values before null analysis
df.loc[df["INSPECTION DATE"] == pd.Timestamp("1900-01-01"), "INSPECTION DATE"] = pd.NaT

In [ ]:
# Replace invalid borough values
df["BORO"] = df["BORO"].replace("0", np.nan)

## 2. Renaming the columns

In [ ]:
df = df.rename(columns={"CAMIS" : "ID", "DBA": "Restaurant Name", "BORO": "Borough"})

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.head(5)

,id,restaurant_name,borough,building,street,zipcode,phone,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type,latitude,longitude
0,50169247,WILDLIFE CONSERVATION SOCIETY,Bronx,2300,SOUTHERN BOULEVARD,10460.0,7187418170,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.850537,-73.882455
1,50180337,BAO'S PASTRY LLC,Queens,135-25,ROOSEVELT AVENUE,11354.0,518526666,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.759150,-73.831345
2,50173810,EARTHBAR,Manhattan,10,EAST 53 STREET,10022.0,6467070033,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.760114,-73.975143
3,50182209,BUTTER CHICKEN SOCIAL,Manhattan,424,5 AVENUE,10018.0,5513259481,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.751049,-73.982665
4,50122548,FAUZIA'S HEAVENLY DELIGHTS INC,Manhattan,517,CLAYTON AVE,NaN,7189305711,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,0.000000,0.000000


## 3. Dropping duplicate rows

In [ ]:
df.shape

(296647, 20)

In [ ]:
duplicate_rows_df = df[df.duplicated()]

print("Number of duplicate rows: ", duplicate_rows_df.shape)

Number of duplicate rows:  (87, 20)


In [ ]:
# Removing exact duplicate data due to it being a small amount (87/297k rows)
df.count()

,0
id,296647
restaurant_name,296645
borough,296487
building,295688
street,296646
zipcode,293468
phone,296633
cuisine_description,293340
inspection_date,293340
action,293340


In [ ]:
df = df.drop_duplicates()
df.head(5)

,id,restaurant_name,borough,building,street,zipcode,phone,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type,latitude,longitude
0,50169247,WILDLIFE CONSERVATION SOCIETY,Bronx,2300,SOUTHERN BOULEVARD,10460.0,7187418170,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.850537,-73.882455
1,50180337,BAO'S PASTRY LLC,Queens,135-25,ROOSEVELT AVENUE,11354.0,518526666,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.759150,-73.831345
2,50173810,EARTHBAR,Manhattan,10,EAST 53 STREET,10022.0,6467070033,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.760114,-73.975143
3,50182209,BUTTER CHICKEN SOCIAL,Manhattan,424,5 AVENUE,10018.0,5513259481,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.751049,-73.982665
4,50122548,FAUZIA'S HEAVENLY DELIGHTS INC,Manhattan,517,CLAYTON AVE,NaN,7189305711,NaN,NaT,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,0.000000,0.000000


## 4. Handle missing/null values

In [ ]:
null_summary = (
    df.isnull().sum()
      .to_frame("null_count")
      .assign(null_pct=lambda x: round(x["null_count"] / len(df) * 100, 2))
      .sort_values("null_count", ascending=False)
)

null_summary

,null_count,null_pct
grade_date,159872,53.91
grade,150342,50.70
score,16631,5.61
violation_description,5606,1.89
violation_code,5605,1.89
inspection_date,3307,1.12
cuisine_description,3307,1.12
action,3307,1.12
inspection_type,3307,1.12
zipcode,3173,1.07


Handling nulls within missing grades (53%) and grade (50%) - some missing values can show inspections that didn't generate a grade. Some rows can occur without a violation detail and a record of an inspection, so we will not drop rows just because these are null.

In [ ]:
# Core columns that have nulls
core_cols = [
    "id", "restaurant_name", "borough", "cuisine_description",
    "inspection_date", "action", "grade", "score", "violation_code"
]

df["core_nulls"] = df[core_cols].isnull().sum(axis=1)
df["core_nulls"].value_counts().sort_index()

,count
core_nulls,
0,145498
1,133479
2,13512
3,764
6,3283
7,24


Most rows have 0 or 1 missing core columns, while a smaller amount has 6 or 7.

In [ ]:
# The colums with >= 6 missing core columns
df[df["core_nulls"] >= 6].head(20)

,id,restaurant_name,borough,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,violation_description,critical_flag,score,grade,grade_date,record_date,inspection_type,latitude,longitude,core_nulls
0,50169247,WILDLIFE CONSERVATION SOCIETY,Bronx,2300,SOUTHERN BOULEVARD,10460.0,7187418170,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.850537,-73.882455,6
1,50180337,BAO'S PASTRY LLC,Queens,135-25,ROOSEVELT AVENUE,11354.0,518526666,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.759150,-73.831345,6
2,50173810,EARTHBAR,Manhattan,10,EAST 53 STREET,10022.0,6467070033,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.760114,-73.975143,6
3,50182209,BUTTER CHICKEN SOCIAL,Manhattan,424,5 AVENUE,10018.0,5513259481,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.751049,-73.982665,6
4,50122548,FAUZIA'S HEAVENLY DELIGHTS INC,Manhattan,517,CLAYTON AVE,NaN,7189305711,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,0.000000,0.000000,6
5,50181054,YOU FU HAPPY BREAKFAST INC.,Queens,40-33,MAIN STREET,11354.0,6469197789,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.758653,-73.829679,6
7,50168375,MEESH TOO LLC,Manhattan,2280,FREDERICK DOUGLASS BOULEVARD,NaN,9175174539,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,0.000000,0.000000,6
8,50160452,TANG HULU,Manhattan,1802,65TH STRETT,NaN,6462567071,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,0.000000,0.000000,6
9,50116692,Be BacSoon,Manhattan,1488,2 AVENUE,10075.0,9175262090,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.772371,-73.955740,6
10,50180814,HOJYOZU NY INC,Manhattan,6,MURRAY STREET,10007.0,6462513399,NaN,NaT,NaN,...,NaN,Not Applicable,NaN,NaN,NaT,2026-03-06,NaN,40.713222,-74.007622,6


A small number of rows are missing most core analytical fields and appear to be incomplete administrative records. These will be retained in the master dataset for transparency, but excluded from analysis-specific subsets.

In [ ]:
# CSV for tableau
df.to_csv("nyc_restaurant_cleaned.csv", index=False)

In [ ]:
pd.set_option('display.max_colwidth', None)

table = df['violation_description'].value_counts().reset_index()
table.columns = ['violation_description', 'count']

display(table.head(50))

,violation_description,count
0,"Non-food contact surface or equipment made of unacceptable material, not kept clean, or not properly sealed, raised, spaced or movable to allow accessibility for cleaning on all sides, above and underneath the unit.",38961
1,"Establishment is not free of harborage or conditions conducive to rodents, insects or other pests.",25307
2,"Food contact surface not properly washed, rinsed and sanitized after each use and following any activity when contamination may have occurred.",18770
3,Anti-siphonage or back-flow prevention device not provided where required; equipment or floor not properly drained; sewage disposal system in disrepair or not functioning properly. Condensation or liquid waste improperly disposed of.,17613
4,Cold TCS food item held above 41 °F; smoked or processed fish held above 38 °F; intact raw eggs held above 45 °F; or reduced oxygen packaged (ROP) TCS foods held above required temperatures except during active necessary preparation.,17402
5,Hot TCS food item not held at or above 140 °F.,14843
6,"Food, supplies, or equipment not protected from potential source of contamination during storage, preparation, transportation, display, service or from customer’s refillable, reusable container. Condiments not in single-service containers or dispensed directly by the vendor.",14731
7,Evidence of mice or live mice in establishment's food or non-food areas.,14384
8,"Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests in establishment’s food and/or non-food areas. FRSA flies include house flies, blow flies, bottle flies, flesh flies, drain flies, Phorid flies and fruit flies.",8983
9,Food Protection Certificate (FPC) not held by manager or supervisor of food operations.,7656


In [ ]:
# Mapping each long violation description to a short category
violation_to_category = {
    "Non-food contact surface or equipment made of unacceptable material, not kept clean, or not properly sealed, raised, spaced or movable to allow accessibility for cleaning on all sides, above and underneath the unit.": "Sanitation",
    "Establishment is not free of harborage or conditions conducive to rodents, insects or other pests.": "Pests",
    "Food contact surface not properly washed, rinsed and sanitized after each use and following any activity when contamination may have occurred.": "Sanitation",
    "Anti-siphonage or back-flow prevention device not provided where required; equipment or floor not properly drained; sewage disposal system in disrepair or not functioning properly. Condensation or liquid waste improperly disposed of.": "Equipment",
    "Cold TCS food item held above 41 °F; smoked or processed fish held above 38 °F; intact raw eggs held above 45 °F; or reduced oxygen packaged (ROP) TCS foods held above required temperatures except during active necessary preparation.": "Temperature",
    "Hot TCS food item not held at or above 140 °F.": "Temperature",
    "Food, supplies, or equipment not protected from potential source of contamination during storage, preparation, transportation, display, service or from customer’s refillable, reusable container. Condiments not in single-service containers or dispensed directly by the vendor.": "Food Safety",
    "Evidence of mice or live mice in establishment's food or non-food areas.": "Pests",
    "Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests in establishment’s food and/or non-food areas. FRSA flies include house flies, blow flies, bottle flies, flesh flies, drain flies, Phorid flies and fruit flies.": "Pests",
    "Food Protection Certificate (FPC) not held by manager or supervisor of food operations.": "Certification",
    "Raw, cooked or prepared food is adulterated, contaminated, cross-contaminated, or not discarded in accordance with HACCP plan.": "Food Safety",
    "Dishwashing and ware washing: Cleaning and sanitizing of tableware, including dishes, utensils, and equipment deficient.": "Utensils",
    "Live roaches in facility's food or non-food area.": "Pests",
    "Sanitized equipment or utensil, including in-use food dispensing utensil, improperly used or stored.": "Sanitation",
    "Wiping cloths not stored clean and dry, or in a sanitizing solution, between uses.": "Sanitation",
    "No hand washing facility in or adjacent to toilet room or within 25 feet of a food preparation, food service or ware washing area. Hand washing facility not accessible, obstructed or used for non-hand washing purposes. No hot and cold running water or water at inadequate pressure. No soap or acceptable hand-drying device.": "Hygiene",
    "Pesticide not properly labeled or used by unlicensed individual. Pesticide, other toxic chemical improperly used/stored. Unprotected, unlocked bait station used.": "Chemicals",
    "Personal cleanliness is inadequate. Outer garment soiled with possible contaminant. Effective hair restraint not worn where required. Jewelry worn on hands or arms. Fingernail polish worn or fingernails not kept clean and trimmed.": "Hygiene",
    "Design, construction, materials used or maintenance of food contact surface improper. Surface not easily cleanable, sanitized and maintained.": "Sanitation",
    "Single service article not provided. Single service article reused or not protected from contamination when transported, stored, dispensed. Drinking straws not completely enclosed in wrapper or dispensed from a sanitary device.": "Food Safety",
    "Evidence of rats or live rats in establishment's food or non-food areas.": "Pests",
    "Thawing procedure improper.": "Temperature",
    "Accurate thermometer not provided or properly located in refrigerated, cold storage or hot holding equipment": "Certification",
    "Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests in establishment’s food and/or non-food areas. FRSA flies include house flies, blow flies, bottle flies, flesh flies, drain flies, Phorid flies and fruit flies.": "Pests",
    "Non-food contact surface improperly constructed. Unacceptable material used. Non-food contact surface or equipment improperly maintained and/or not properly sealed, raised, spaced or movable to allow accessibility for cleaning on all sides, above and underneath the unit.": "Sanitation",
    "Wash hands sign not posted near or above hand washing sink.": "Signage",
    "Properly scaled and calibrated thermometer or thermocouple not provided or not readily accessible in food preparation and hot/cold holding areas to measure temperatures of TCS foods during cooking, cooling, reheating, and holding.": "Certification",
    "After cooking or removal from hot holding, TCS food not cooled by an approved method whereby the internal temperature is reduced from 140 °F to 70 °F or less within 2 hours, and from 70 °F to 41 °F or less within 4 additional hours.": "Temperature",
    "Current letter grade or Grade Pending card not posted": "Signage",
    "“Choking first aid” poster not posted. “Alcohol and Pregnancy” warning sign not posted. Resuscitation equipment: exhaled air resuscitation masks (adult & pediatric), latex gloves, sign not posted.": "Signage",
    "Food, supplies, and equipment not protected from potential source of contamination during storage, preparation, transportation, display or service.": "Food Safety",
    "Failure to post or conspicuously post healthy eating information": "Signage",
    "Facility not vermin proof. Harborage or conditions conducive to attracting vermin to the premises and/or allowing vermin to exist.": "Pests",
    "Contract with a pest management professional not in place. Record of extermination activities not kept on premises.": "Pests",
    "Mechanical or natural ventilation not provided, inadequate, improperly installed, in disrepair or fails to prevent and control excessive build-up of grease, heat, steam condensation, vapors, odors, smoke or fumes.": "Equipment",
    "Garbage receptacle not pest or water resistant, with tight-fitting lids, and covered except while in active use. Garbage receptacle and cover not cleaned after emptying and prior to reuse. Garbage, refuse and other solid and liquid waste not collected, stored, removed and disposed of so as to prevent a nuisance.": "Sanitation",
    "Evidence of mice or live mice present in facility's food and/or non-food areas.": "Pests",
    "Tobacco or electronic cigarette use, eating, or drinking from open container in food preparation, food storage or dishwashing area.": "Hygiene",
    "Food worker/food vendor does not use utensil or other barrier to eliminate bare hand contact with food that will not receive adequate additional heat treatment.": "Food Safety",
    "Toilet facility not maintained or provided with toilet paper, waste receptacle or self-closing door.": "Hygiene",
    "Providing single-use, non-compostable plastic straws to customers without customer request (including providing such straws at a self-serve station)": "Food Safety",
    "Food, prohibited, from unapproved or unknown source, home canned or home prepared. Animal slaughtered, butchered or dressed (eviscerated, skinned) in establishment. Reduced Oxygen Packaged (ROP) fish not frozen before processing. ROP food prepared on premises transported to another site.": "Food Safety",
    "No approved written standard operating procedure for avoiding contamination by refillable returnable containers.": "Food Safety",
    "Dishwashing and ware washing: Cleaning and sanitizing of tableware, including dishes, utensils, and equipment deficient.": "Utensils",
    "Food not protected from potential source of contamination during storage, preparation, transportation, display or service.": "Food Safety",
    "Pesticide not properly labeled or used by unlicensed individual. Pesticide, other toxic chemical improperly used/stored. Unprotected, unlocked bait station used.": "Chemicals",
    "Food allergy information poster not conspicuously posted where food is being prepared or processed by food workers.": "Signage",
    "Cold food item held above 41º F (smoked fish and reduced oxygen packaged foods above 38 ºF) except during necessary preparation.": "Temperature",
    "Plumbing not properly installed or maintained; anti-siphonage or backflow prevention device not provided where required; equipment or floor not properly drained; sewage disposal system in disrepair or not functioning properly.": "Equipment",
    "Insufficient or no hot holding, cold storage or cold holding equipment provided to maintain Time/Temperature Control for Safety Foods (TCS) at required temperatures": "Temperature"
}

In [ ]:
# Add a new column 'category' to the original DataFrame
df['category'] = df['violation_description'].map(violation_to_category)
df['category'] = df['category'].fillna('Other')
df['category'].value_counts()

,count
category,
Sanitation,74753
Pests,61178
Temperature,38750
Food Safety,30792
Other,29619
Equipment,20015
Certification,12000
Hygiene,10565
Signage,8847


In [ ]:
# CSV for tableau
df.to_csv("nyc_restaurant_cleaned_2.csv", index=False)

In [ ]:
# Short category labels and summaries
category_summary = {
    "Sanitation": "Dirty or improperly maintained surfaces and equipment",
    "Pests": "Rodents, mice, roaches, flies, or other nuisance pests",
    "Temperature": "Food not held at proper temperatures (hot/cold TCS foods)",
    "Food Safety": "Food, supplies, or utensils not protected from contamination",
    "Hygiene": "Hand washing, personal cleanliness, or toilet facilities issues",
    "Certification": "Missing required certifications or thermometers",
    "Utensils": "Dishwashing or utensil cleaning deficiencies",
    "Signage": "Missing required posters, signs, or grade cards",
    "Equipment": "Plumbing, ventilation, or other facility equipment issues",
    "Chemicals": "Improper use or storage of pesticides/chemicals"
}

# Convert to DataFrame
category_df = pd.DataFrame.from_dict(
    category_summary,
    orient='index',
    columns=['Summary']
).reset_index().rename(columns={'index': 'Category'})

# Display nicely
display(category_df)

,Category,Summary
0,Sanitation,Dirty or improperly maintained surfaces and equipment
1,Pests,"Rodents, mice, roaches, flies, or other nuisance pests"
2,Temperature,Food not held at proper temperatures (hot/cold TCS foods)
3,Food Safety,"Food, supplies, or utensils not protected from contamination"
4,Hygiene,"Hand washing, personal cleanliness, or toilet facilities issues"
5,Certification,Missing required certifications or thermometers
6,Utensils,Dishwashing or utensil cleaning deficiencies
7,Signage,"Missing required posters, signs, or grade cards"
8,Equipment,"Plumbing, ventilation, or other facility equipment issues"
9,Chemicals,Improper use or storage of pesticides/chemicals
